In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

In [13]:
from pathlib import Path
import pandas as pd

input_path = Path(r"D:\Suwaaq\College\Sem 5\ML\ML_2026\data\processed\olist_orders_abt.csv")

print("Resolved path:", input_path.resolve())
print("Exists:", input_path.exists())

df = pd.read_csv(input_path)

print("Shape:", df.shape)
df.head()

Resolved path: D:\Suwaaq\College\Sem 5\ML\ML_2026\data\processed\olist_orders_abt.csv
Exists: True
Shape: (99441, 29)


,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_year,order_month,order_day,order_day_of_week,...,total_payment_value,max_payment_installments,payment_types_count,dominant_payment_type,total_items,total_price,total_freight,unique_products,unique_sellers,main_product_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017,10,2,0,...,38.71,1.0,2.0,voucher,1.0,29.99,8.72,1.0,1.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018,7,24,1,...,141.46,1.0,1.0,boleto,1.0,118.70,22.76,1.0,1.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018,8,8,2,...,179.12,3.0,1.0,credit_card,1.0,159.90,19.22,1.0,1.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,delivered,2017,11,18,5,...,72.20,1.0,1.0,credit_card,1.0,45.00,27.20,1.0,1.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018,2,13,1,...,28.62,1.0,1.0,credit_card,1.0,19.90,8.72,1.0,1.0,stationery


In [14]:
target = "is_late_delivery"

In [15]:
identifier_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

In [16]:
leakage_columns = [
    "delivery_days",
    "delivery_delay_days",
    "review_score",
    "review_comment_count",
    "has_review_comment",
    "is_low_review"
]

In [17]:
columns_to_drop = [
    c for c in identifier_columns + leakage_columns
    if c in df.columns
]

feature_df = df.drop(columns=columns_to_drop)

print(feature_df.columns.tolist())

['customer_city', 'customer_state', 'order_status', 'order_year', 'order_month', 'order_day', 'order_day_of_week', 'order_hour', 'estimated_delivery_days', 'is_late_delivery', 'total_payment_value', 'max_payment_installments', 'payment_types_count', 'dominant_payment_type', 'total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'main_product_category']


In [18]:
if {"total_price", "total_items"}.issubset(df.columns):
    df["average_item_price"] = (df["total_price"] / df["total_items"].replace(0, np.nan))

In [ ]:
if {"total_freight", "total_price"}.issubset(df.columns):
    df["freight_ratio"] = (
        df["total_freight"] /
        df["total_price"].replace(0, np.nan)
    )

In [20]:
if {"total_items", "unique_sellers"}.issubset(df.columns):
    df["items_per_seller"] = (
        df["total_items"] /
        df["unique_sellers"].replace(0, np.nan)
    )

In [21]:
if {"unique_sellers", "total_items"}.issubset(df.columns):
    df["seller_diversity"] = (
        df["unique_sellers"] /
        df["total_items"].replace(0, np.nan)
    )

In [22]:
if "order_day_of_week" in df.columns:
    df["is_weekend"] = (
        df["order_day_of_week"] >= 5
    ).astype(int)

In [23]:
if "order_hour" in df.columns:
    df["is_business_hour"] = (
        df["order_hour"].between(9, 18)
    ).astype(int)

In [24]:
if "order_month" in df.columns:
    df["is_year_end"] = (
        df["order_month"].isin([11, 12])
    ).astype(int)

In [25]:
if "order_month" in df.columns:
    df["month_sin"] = np.sin(
        2 * np.pi * df["order_month"] / 12
    )

    df["month_cos"] = np.cos(
        2 * np.pi * df["order_month"] / 12
    )

In [26]:
if "order_hour" in df.columns:
    df["hour_sin"] = np.sin(
        2 * np.pi * df["order_hour"] / 24
    )
    
    df["hour_cos"] = np.cos(
        2 * np.pi * df["order_hour"] / 24
    )

In [27]:
if "total_price" in df.columns:
    df["log_total_price"] = np.log1p(
        df["total_price"].clip(lower=0)
    )

if "total_freight" in df.columns:
    df["log_total_freight"] = np.log1p(
        df["total_freight"].clip(lower=0)
    )


In [28]:
if "total_price" in df.columns:
    df["price_band"] = pd.cut(
        df["total_price"],
            bins=[-np.inf, 100, 500, 1000, 5000, np.inf],
            labels=[
            "very_low",
            "low",
            "medium",
            "high",
            "very_high"
        ]
    )

In [29]:
if {"total_items", "total_price"}.issubset(df.columns):
    df["items_x_price"] = (
        df["total_items"] * df["total_price"]
    )

In [31]:
engineered_columns = [
    "average_item_price",
    "freight_ratio",
    "items_per_seller",
    "seller_diversity"
]

available_engineered = [
    c for c in engineered_columns
    if c in df.columns
]

df[available_engineered].isna().sum()

average_item_price    775
freight_ratio         775
items_per_seller      775
seller_diversity      775
dtype: int64

In [32]:
new_columns = [
    "average_item_price",
    "freight_ratio",
    "items_per_seller",
    "seller_diversity",
    "is_weekend",
    "is_business_hour",
    "is_year_end",
    "month_sin",
    "month_cos",
    "hour_sin",
    "hour_cos",
    "log_total_price",
    "log_total_freight",
    "price_band",
    "items_x_price"
]

In [33]:
new_columns = [
    c for c in new_columns
    if c in df.columns
]

df[new_columns].head()

,average_item_price,freight_ratio,items_per_seller,seller_diversity,is_weekend,is_business_hour,is_year_end,month_sin,month_cos,hour_sin,hour_cos,log_total_price,log_total_freight,price_band,items_x_price
0,29.99,0.290764,1.0,1.0,0,1,0,-0.866025,0.500000,0.500000,-0.866025,3.433665,2.274186,very_low,29.99
1,118.70,0.191744,1.0,1.0,0,0,0,-0.500000,-0.866025,-0.866025,0.500000,4.784989,3.168003,low,118.70
2,159.90,0.120200,1.0,1.0,0,0,0,-0.866025,-0.500000,0.866025,-0.500000,5.080783,3.006672,low,159.90
3,45.00,0.604444,1.0,1.0,1,0,1,-0.500000,0.866025,-0.965926,0.258819,3.828641,3.339322,very_low,45.00
4,19.90,0.438191,1.0,1.0,0,0,0,0.866025,0.500000,-0.707107,0.707107,3.039749,2.274186,very_low,19.90


In [34]:
from sklearn.feature_selection import VarianceThreshold

In [36]:
numeric_df = df.select_dtypes(
    include=np.number
).copy()

numeric_df = numeric_df.drop(
    columns=[target],
    errors="ignore"
)

selector = VarianceThreshold(threshold=0.0)

selector.fit(
    numeric_df.fillna(numeric_df.median())
)

selected_numeric = numeric_df.columns[
    selector.get_support()
]

print("Original:", len(numeric_df.columns))
print("After variance filtering:", len(selected_numeric))

Original: 34
After variance filtering: 34


In [37]:
corr = numeric_df.corr(numeric_only=True)
corr

,order_year,order_month,order_day,order_day_of_week,order_hour,delivery_days,estimated_delivery_days,delivery_delay_days,review_score,is_low_review,...,is_weekend,is_business_hour,is_year_end,month_sin,month_cos,hour_sin,hour_cos,log_total_price,log_total_freight,items_x_price
order_year,1.000000,-0.550059,-0.043672,-0.017917,-0.003229,-0.050156,-0.142717,0.067070,0.004872,0.000559,...,-0.007842,0.014861,-0.418601,0.384586,-0.239787,0.002193,-0.018217,0.002420,0.006291,-0.001697
order_month,-0.550059,1.000000,0.001343,0.020588,-0.003953,-0.054398,-0.094182,0.025640,0.027560,-0.023095,...,0.006949,-0.013991,0.653618,-0.779053,0.104784,0.005060,0.013300,0.007743,0.008077,0.002842
order_day,-0.043672,0.001343,1.000000,-0.024895,-0.012195,-0.000528,-0.030391,0.024665,0.001774,-0.001763,...,-0.012283,0.004332,0.049311,-0.007958,0.040667,0.008465,-0.005243,-0.012869,-0.006626,0.001063
order_day_of_week,-0.017917,0.020588,-0.024895,1.000000,0.009433,0.028807,0.067685,-0.031395,-0.008246,0.003517,...,0.768693,-0.020495,0.036749,-0.002184,0.021455,-0.027702,0.030770,-0.007037,0.001661,-0.004792
order_hour,-0.003229,-0.003953,-0.012195,0.009433,1.000000,-0.004758,-0.000144,0.022651,0.005765,-0.006700,...,0.043324,-0.284269,-0.002924,0.001749,-0.002273,-0.731042,0.441757,0.009325,0.004170,-0.003033
delivery_days,-0.050156,-0.054398,-0.000528,0.028807,-0.004758,1.000000,0.382836,0.607716,-0.334150,0.305947,...,0.003208,0.008565,0.110843,0.139792,0.200153,-0.004024,-0.004807,0.084508,0.252799,0.022864
estimated_delivery_days,-0.142717,-0.094182,-0.030391,0.067685,-0.000144,0.382836,1.000000,-0.499649,-0.054533,0.042413,...,0.038642,-0.004614,0.042641,0.158750,0.130545,-0.005420,0.008675,0.119597,0.369335,0.039232
delivery_delay_days,0.067070,0.025640,0.024665,-0.031395,0.022651,0.607716,-0.499649,1.000000,-0.266990,0.252709,...,-0.030329,-0.005406,0.063323,-0.001307,0.075595,-0.018859,0.010679,-0.023462,-0.082287,-0.012812
review_score,0.004872,0.027560,0.001774,-0.008246,0.005765,-0.334150,-0.054533,-0.266990,1.000000,-0.883055,...,-0.000856,0.000969,-0.037643,-0.057687,-0.077579,0.003125,-0.004266,-0.046397,-0.103214,-0.056734
is_low_review,0.000559,-0.023095,-0.001763,0.003517,-0.006700,0.305947,0.042413,0.252709,-0.883055,1.000000,...,-0.001134,-0.001281,0.033487,0.049125,0.067118,-0.001755,0.003539,0.053767,0.096209,0.057669


In [38]:
threshold = 0.90
upper = corr.where(
    np.triu(
    np.ones(corr.shape),
    k=1
    ).astype(bool)
)

high_corr_pairs = []
for col in upper.columns:
    for row in upper.index:
        value = upper.loc[row, col]

        if pd.notna(value) and abs(value) > threshold:
            high_corr_pairs.append(
                (row, col, value)
            )
            
            high_corr_pairs[:20]

In [39]:
from sklearn.feature_selection import mutual_info_classif

In [40]:
mi_df = df.select_dtypes(
    include=np.number
).copy()

mi_df = mi_df.drop(
    columns=[target],
    errors="ignore"
)

mi_df = mi_df.fillna(
    mi_df.median()
)

mi_scores = mutual_info_classif(
    mi_df,
    df[target],
    random_state=42
)

mi_results = pd.DataFrame({
    "feature": mi_df.columns,
    "mutual_information": mi_scores
}).sort_values(
    "mutual_information",
    ascending=False
)

mi_results.head(15)

,feature,mutual_information
7,delivery_delay_days,0.243227
5,delivery_days,0.133898
8,review_score,0.052955
9,is_low_review,0.041675
19,unique_sellers,0.027428
18,unique_products,0.023377
25,is_business_hour,0.017332
14,payment_types_count,0.016252
23,seller_diversity,0.014488
1,order_month,0.013751


In [41]:
from sklearn.ensemble import RandomForestClassifier

In [42]:
X = df.select_dtypes(
    include=np.number
).drop(
    columns=[target],
    errors="ignore"
)

X = X.fillna(X.median())
y = df[target]

In [43]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf.fit(X, y)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [44]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

importance.head(15)

,feature,importance
7,delivery_delay_days,0.527008
5,delivery_days,0.252009
8,review_score,0.077646
9,is_low_review,0.051059
6,estimated_delivery_days,0.024140
28,month_cos,0.009838
1,order_month,0.008549
27,month_sin,0.006220
32,log_total_freight,0.005879
17,total_freight,0.005191


In [45]:
candidate_features = [
    "total_price",
    "total_freight",
    "total_items",
    "unique_products",
    "unique_sellers",
    "order_month",
    "order_day_of_week",
    "order_hour",
    "average_item_price",
    "freight_ratio",
    "items_per_seller",
    "seller_diversity",
    "is_weekend",
    "is_business_hour",
    "month_sin",
    "month_cos",
    "hour_sin",
    "hour_cos",
    "log_total_price",
    "log_total_freight",
    "items_x_price"
]

final_features = [
    c for c in candidate_features
    if c in df.columns
]

final_df = df[
    final_features + [target]
].copy()

print("Final shape:", final_df.shape)
final_df.head()

Final shape: (99441, 22)


,total_price,total_freight,total_items,unique_products,unique_sellers,order_month,order_day_of_week,order_hour,average_item_price,freight_ratio,...,is_weekend,is_business_hour,month_sin,month_cos,hour_sin,hour_cos,log_total_price,log_total_freight,items_x_price,is_late_delivery
0,29.99,8.72,1.0,1.0,1.0,10,0,10,29.99,0.290764,...,0,1,-0.866025,0.500000,0.500000,-0.866025,3.433665,2.274186,29.99,0
1,118.70,22.76,1.0,1.0,1.0,7,1,20,118.70,0.191744,...,0,0,-0.500000,-0.866025,-0.866025,0.500000,4.784989,3.168003,118.70,0
2,159.90,19.22,1.0,1.0,1.0,8,2,8,159.90,0.120200,...,0,0,-0.866025,-0.500000,0.866025,-0.500000,5.080783,3.006672,159.90,0
3,45.00,27.20,1.0,1.0,1.0,11,5,19,45.00,0.604444,...,1,0,-0.500000,0.866025,-0.965926,0.258819,3.828641,3.339322,45.00,0
4,19.90,8.72,1.0,1.0,1.0,2,1,21,19.90,0.438191,...,0,0,0.866025,0.500000,-0.707107,0.707107,3.039749,2.274186,19.90,0


In [46]:
output_dir = Path("data/processed")
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    output_dir /
    "olist_orders_feature_engineered.csv"
)

final_df.to_csv(
    output_path,
    index=False
)

print("Saved to:", output_path)

Saved to: data\processed\olist_orders_feature_engineered.csv
